# Customer Activity and Churn Definition

This notebook first constructs a cleaned customer-level transaction history from the two raw worksheets. It then studies purchasing behavior to support an evidence-based inactivity definition. Each cleaning decision is inherited from the raw-data audit and remains explicit; no churn threshold is assumed at this stage.

## 1. Combined raw transaction table

The two Excel worksheets are stacked into one table. `SourceSheet` preserves the origin of each row. No overlap reconciliation, duplicate removal, filtering, or cleaning is performed in this step.

In [ ]:
import pandas as pd

workbook = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name=None)
transactions_raw = pd.concat(
    [transactions.assign(SourceSheet=sheet_name) for sheet_name, transactions in workbook.items()],
    ignore_index=True,
)
transactions_raw.shape

## 2. Exact duplicate removal

Exact duplicates are identified using the eight original columns. `SourceSheet` is excluded from the comparison so that identical rows from the worksheet overlap are also detected. The first occurrence is retained.

In [ ]:
original_columns = transactions_raw.columns.drop("SourceSheet")
transactions_clean = transactions_raw.drop_duplicates(subset=original_columns).copy()
transactions_clean.shape

## 3. Negative-price accounting adjustments

The audit found that all negative-price rows are bad-debt accounting adjustments rather than customer purchases. They are removed from `transactions_clean`. Zero-price rows remain unchanged because customer-linked free items may still represent genuine activity.

In [ ]:
negative_price_rows = transactions_clean["Price"] < 0
print("Rows removed:", negative_price_rows.sum())
transactions_clean = transactions_clean.loc[~negative_price_rows].copy()
transactions_clean.shape

## 4. Identified customer transactions

Customer-level analysis requires a known `Customer ID`. Anonymous rows are therefore excluded from `customer_transactions`, while `transactions_clean` remains unchanged as the broader cleaned transaction history. After this filter, no missing values remain in `customer_transactions`. No quantity, cancellation, manual-code, or zero-price filter is applied here.

In [ ]:
customer_transactions = transactions_clean.dropna(subset=["Customer ID"]).copy()
print("Rows removed:", len(transactions_clean) - len(customer_transactions))
customer_transactions.shape

## 5. Remaining numerical extremes

The most extreme quantities and prices are inspected with their transaction context before any treatment is considered. This review distinguishes unusual values from demonstrated errors and does not modify the data.

In [ ]:
inspection_columns = ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID"]
display(customer_transactions[["Quantity", "Price"]].describe())
print("Lowest quantities")
display(customer_transactions.nsmallest(10, "Quantity")[inspection_columns])
print("Highest quantities")
display(customer_transactions.nlargest(10, "Quantity")[inspection_columns])
print("Highest prices")
display(customer_transactions.nlargest(10, "Price")[inspection_columns])

### 5.1 Interpretation of retained extremes

Negative quantities are retained because cancellation invoices beginning with `C` represent customer interactions, even though they are not new purchases. Rows with `StockCode = M` are also retained because `Manual` is ambiguous: it may represent a manually entered order, charge, or correction. These records will be separated by behavioral role later rather than deleted during cleaning. No positive extreme is removed without evidence that it is erroneous.

## 6. Identifier data types

`Customer ID`, `Invoice`, and `StockCode` are identifiers rather than numerical measures. They are converted to pandas string values for consistent grouping and code-based filtering. This conversion does not remove or otherwise modify any records.

In [ ]:
customer_transactions["Customer ID"] = customer_transactions["Customer ID"].astype("int64").astype("string")
customer_transactions["Invoice"] = customer_transactions["Invoice"].astype("string")
customer_transactions["StockCode"] = customer_transactions["StockCode"].astype("string")
customer_transactions[["Customer ID", "Invoice", "StockCode"]].dtypes

## 7. Test transaction removal

`TEST001` and `TEST002` are explicitly described as test products. Their 16 rows are isolated in `test_transactions` for traceability and excluded from the customer history because they do not represent confirmed commercial activity.

In [ ]:
test_rows = customer_transactions["StockCode"].isin(["TEST001", "TEST002"])
test_transactions = customer_transactions.loc[test_rows].copy()
customer_transactions = customer_transactions.loc[~test_rows].copy()
print("Test rows removed:", len(test_transactions))
customer_transactions.shape

## 8. Text normalization

`Description` and `Country` are converted to pandas strings, and surrounding whitespace is removed. Original spelling and capitalization are preserved because no country remapping or product-description standardization has been approved.

In [ ]:
customer_transactions["Description"] = customer_transactions["Description"].astype("string").str.strip()
customer_transactions["Country"] = customer_transactions["Country"].astype("string").str.strip()
customer_transactions[["Description", "Country"]].dtypes

## 9. Final temporal structure

Calendar `Year` is derived from each original `InvoiceDate`. `SourceSheet` is removed because it records workbook provenance rather than transaction time. The original timestamps are preserved, the rows are sorted chronologically, and the index is reset.

In [ ]:
customer_transactions["Year"] = customer_transactions["InvoiceDate"].dt.year.astype("int16")
customer_transactions = (
    customer_transactions.drop(columns="SourceSheet")
    .sort_values("InvoiceDate")
    .reset_index(drop=True)
)
customer_transactions.head()

## 10. Multi-timestamp invoice decision

The audit identified 65 invoices with two or more timestamps separated by only one or two minutes. Inspection showed batches of different product lines recorded under the same invoice, customer, and country. This pattern is compatible with a single invoice entered in consecutive batches, but it does not prove the underlying process. The line-level timestamps therefore remain unchanged. For invoice-level event timing, the approved convention uses the final observed timestamp to represent completion of the invoice entry.

## 11. Cleaning validation

The final checks confirm the approved rules. Missing values, exact duplicates, negative prices, zero quantities, test rows, and surrounding description spaces should be absent. Negative quantities, cancellation invoices, manual records, zero-price customer activity, and the 65 multi-timestamp invoices remain intentionally preserved.

In [ ]:
print("Rows:", len(customer_transactions))
print("Missing values:", customer_transactions.isna().sum().sum())
print("Exact duplicates:", customer_transactions.duplicated(subset=original_columns).sum())
print("Negative prices:", (customer_transactions["Price"] < 0).sum())
print("Zero quantities:", (customer_transactions["Quantity"] == 0).sum())
print("Test rows retained:", customer_transactions["StockCode"].isin(["TEST001", "TEST002"]).sum())
print("Description spaces:", customer_transactions["Description"].ne(customer_transactions["Description"].str.strip()).sum())
print("Multi-date invoices:", (customer_transactions.groupby(["Customer ID", "Invoice"])["InvoiceDate"].nunique() > 1).sum())
print("Negative quantities retained:", (customer_transactions["Quantity"] < 0).sum())
print("C invoice rows retained:", customer_transactions["Invoice"].str.startswith("C").sum())
print("Manual rows retained:", (customer_transactions["StockCode"] == "M").sum())
print("Zero-price rows retained:", (customer_transactions["Price"] == 0).sum())

## 12. Interim dataset export

The validated line-level customer history is exported as Parquet in `data/interim/` with its original timestamps. It remains an interim dataset because purchase-event definitions and churn rules have not yet been approved.

In [ ]:
output_path = "../data/interim/customer_transactions.parquet"
customer_transactions.to_parquet(output_path, index=False)
print("Saved:", output_path)

## 13. Invoice-level analytical table

The cleaned line-level history is aggregated into `invoice_events`, where one row represents one invoice for one customer. `InvoiceDate` is defined as the latest timestamp observed within the invoice, representing completion of the invoice entry. The country, line and product counts, total quantity, cancellation status, presence of manual lines, manual-only status, and presence of zero-price lines are retained. These indicators classify events without removing them or defining churn. Original timestamps and product-level detail remain available in `customer_transactions`.

In [ ]:
invoice_events = (
    customer_transactions.groupby(["Invoice", "Customer ID"])
    .agg(
        InvoiceDate=("InvoiceDate", "max"),
        Country=("Country", "first"),
        LineCount=("StockCode", "size"),
        ProductCount=("StockCode", "nunique"),
        TotalQuantity=("Quantity", "sum"),
        HasManual=("StockCode", lambda codes: codes.eq("M").any()),
        ManualOnly=("StockCode", lambda codes: codes.eq("M").all()),
        HasZeroPrice=("Price", lambda prices: prices.eq(0).any()),
    )
    .reset_index()
)
invoice_events["IsCancellation"] = invoice_events["Invoice"].str.startswith("C")
print("Invoice events:", len(invoice_events))
display(invoice_events.head())
display(invoice_events[["IsCancellation", "HasManual", "ManualOnly", "HasZeroPrice"]].value_counts())

## 14. Positive transactions versus all interactions

Customer activity is compared under two descriptive definitions. Positive activity includes invoices not beginning with `C`, while all interactions also include cancellations. For each customer, the comparison measures invoice counts, the latest date under each definition, and the number of days by which a later cancellation extends the observed interaction history. Positive manual invoices remain included at this exploratory stage. These full-history summaries are not modeling features or churn labels.

In [ ]:
positive_events = invoice_events.loc[~invoice_events["IsCancellation"]].copy()
positive_activity = positive_events.groupby("Customer ID").agg(
    PositiveInvoices=("Invoice", "count"),
    LastPositiveDate=("InvoiceDate", "max"),
)
all_activity = invoice_events.groupby("Customer ID").agg(
    AllInteractions=("Invoice", "count"),
    LastInteractionDate=("InvoiceDate", "max"),
)
customer_activity_comparison = all_activity.join(positive_activity, how="left")
customer_activity_comparison["PositiveInvoices"] = customer_activity_comparison["PositiveInvoices"].fillna(0).astype("int64")
customer_activity_comparison["CancellationInvoices"] = customer_activity_comparison["AllInteractions"] - customer_activity_comparison["PositiveInvoices"]
customer_activity_comparison["InteractionExtensionDays"] = (customer_activity_comparison["LastInteractionDate"] - customer_activity_comparison["LastPositiveDate"]).dt.days
print("Customers:", len(customer_activity_comparison))
print("Customers without positive transactions:", customer_activity_comparison["LastPositiveDate"].isna().sum())
print("Customers with a later cancellation:", (customer_activity_comparison["InteractionExtensionDays"] > 0).sum())
display(customer_activity_comparison[["PositiveInvoices", "CancellationInvoices", "InteractionExtensionDays"]].describe())
display(customer_activity_comparison.nlargest(10, "InteractionExtensionDays"))

### 14.1 Eligibility decision for purchase-based churn analysis

The comparison identifies 61 customers with cancellation invoices but no observed positive transaction. They remain in `customer_transactions` and `invoice_events` for traceability, but they will be excluded from the future purchase-based churn population because no initial purchase is available from which to measure recurrence or inactivity. Under the current inclusive definition, in which every non-`C` invoice including positive manual invoices counts as a positive transaction candidate, 5,879 customers are eligible. This count must be recalculated if the treatment of manual-only invoices changes.

## 15. Customer recurrence

Recurrence is measured among customers with at least one positive transaction candidate. For each eligible customer, the number of positive invoices, first and last positive transaction dates, and elapsed days between those dates are calculated. Customers with one invoice are separated from recurring customers. Cancellation invoices are excluded from these measures, while positive manual invoices remain included under the current exploratory definition.

In [ ]:
customer_recurrence = positive_events.groupby("Customer ID").agg(
    PositiveInvoices=("Invoice", "count"),
    FirstPositiveDate=("InvoiceDate", "min"),
    LastPositiveDate=("InvoiceDate", "max"),
)
customer_recurrence["ObservedActivityDays"] = (
    customer_recurrence["LastPositiveDate"] - customer_recurrence["FirstPositiveDate"]
).dt.days
one_time_customers = customer_recurrence["PositiveInvoices"] == 1
print("Eligible customers:", len(customer_recurrence))
print("One-time customers:", one_time_customers.sum(), f"({one_time_customers.mean():.1%})")
print("Recurring customers:", (~one_time_customers).sum(), f"({(~one_time_customers).mean():.1%})")
display(customer_recurrence[["PositiveInvoices", "ObservedActivityDays"]].describe())

## 16. Inter-purchase intervals

For recurring customers, the elapsed days between consecutive positive invoices are calculated from the invoice-level table. The distribution of all intervals gives more weight to frequent customers, so a second summary calculates each customer's median, mean, and maximum interval. Cancellation invoices are excluded, while positive manual invoices remain included under the current exploratory definition. No inactivity or churn threshold is selected in this step.

In [ ]:
positive_events_sorted = positive_events.sort_values(["Customer ID", "InvoiceDate"]).copy()
positive_events_sorted["InterpurchaseDays"] = (
    positive_events_sorted.groupby("Customer ID")["InvoiceDate"].diff().dt.total_seconds() / 86400
)
interpurchase_gaps = positive_events_sorted.dropna(subset=["InterpurchaseDays"]).copy()
customer_gap_summary = interpurchase_gaps.groupby("Customer ID")["InterpurchaseDays"].agg(
    GapCount="count",
    MedianGapDays="median",
    MeanGapDays="mean",
    MaxGapDays="max",
)
print("Inter-purchase intervals:", len(interpurchase_gaps))
display(interpurchase_gaps["InterpurchaseDays"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
display(customer_gap_summary[["MedianGapDays", "MeanGapDays", "MaxGapDays"]].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

## 17. Candidate inactivity-threshold comparison

Candidate thresholds from 30 days to one year are compared using completed inter-purchase gaps. For each threshold, the table reports how many observed gaps exceed it and how many recurring customers are affected. The 365-day threshold is retained explicitly to inspect annual or seasonal purchasing behavior. These customers eventually returned, so long completed gaps reveal where an inactivity rule could trigger before a later purchase. Cancellation invoices remain available as interactions but do not reset the purchase interval. This comparison does not create a churn label or select a final threshold.

In [ ]:
thresholds = [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 365]
threshold_comparison = pd.DataFrame({
    "ThresholdDays": thresholds,
    "GapsAbove": [(interpurchase_gaps["InterpurchaseDays"] > threshold).sum() for threshold in thresholds],
    "GapPercentage": [(interpurchase_gaps["InterpurchaseDays"] > threshold).mean() * 100 for threshold in thresholds],
    "CustomersAffected": [interpurchase_gaps.loc[interpurchase_gaps["InterpurchaseDays"] > threshold, "Customer ID"].nunique() for threshold in thresholds],
})
threshold_comparison["CustomerPercentage"] = (
    threshold_comparison["CustomersAffected"] / len(customer_gap_summary) * 100
)
threshold_comparison.round(2)

### 17.1 Comparison when cancellation invoices count as interactions

To measure the effect of cancellation invoices, the same threshold analysis is repeated using every invoice event, including invoices beginning with `C`. The comparison is restricted to the same recurring customers used in the purchase-gap analysis, so the customer denominator remains unchanged. Under this alternative definition, a cancellation resets the interaction interval. This does not replace the purchase-based definition because a cancellation is customer activity but not a new purchase.

In [ ]:
recurring_customer_ids = customer_gap_summary.index
interaction_events = (
    invoice_events.loc[invoice_events["Customer ID"].isin(recurring_customer_ids)]
    .sort_values(["Customer ID", "InvoiceDate"])
    .copy()
)
interaction_events["InteractionGapDays"] = (
    interaction_events.groupby("Customer ID")["InvoiceDate"].diff().dt.total_seconds() / 86400
)
interaction_gaps = interaction_events.dropna(subset=["InteractionGapDays"])

comparison_with_c = threshold_comparison[["ThresholdDays", "GapPercentage", "CustomerPercentage"]].copy()
comparison_with_c = comparison_with_c.rename(columns={
    "GapPercentage": "PurchaseGapPercentage",
    "CustomerPercentage": "PurchaseCustomerPercentage",
})
comparison_with_c["InteractionGapPercentage"] = [
    (interaction_gaps["InteractionGapDays"] > threshold).mean() * 100 for threshold in thresholds
]
comparison_with_c["InteractionCustomerPercentage"] = [
    interaction_gaps.loc[interaction_gaps["InteractionGapDays"] > threshold, "Customer ID"].nunique()
    / len(recurring_customer_ids) * 100
    for threshold in thresholds
]
comparison_with_c.round(2)

## 18. Return probability after prolonged inactivity

For each candidate inactivity threshold, this analysis measures the probability of a later positive purchase within an additional 30, 60, 90, or 180 days. Completed inter-purchase periods are combined with the censored period between each customer's final purchase and the end of observation. A censored period is included for a given return horizon only when enough follow-up time is available; otherwise, its outcome is unknown and it is excluded from that denominator. The table reports both the periods that reached the threshold and those with an observable outcome, because high thresholds have less supporting data. Cancellation invoices do not count as returns. Frequent customers can contribute several completed periods, so the result describes inactivity periods rather than equally weighted customers.

In [61]:
observation_end = positive_events["InvoiceDate"].max()

completed_spells = interpurchase_gaps[["Customer ID", "InterpurchaseDays"]].rename(
    columns={"InterpurchaseDays": "DurationDays"}
)
completed_spells["Returned"] = True

terminal_spells = customer_recurrence[["LastPositiveDate"]].reset_index()
terminal_spells["DurationDays"] = (
    observation_end - terminal_spells["LastPositiveDate"]
).dt.total_seconds() / 86400
terminal_spells["Returned"] = False
terminal_spells = terminal_spells[["Customer ID", "DurationDays", "Returned"]]

inactivity_spells = pd.concat([completed_spells, terminal_spells], ignore_index=True)
results = []

return_horizons = [30, 60, 90, 180]

for threshold in thresholds:
    reached_threshold = inactivity_spells.loc[inactivity_spells["DurationDays"] > threshold]
    for horizon in return_horizons:
        observable = reached_threshold.loc[
            reached_threshold["Returned"]
            | (reached_threshold["DurationDays"] >= threshold + horizon)
        ]
        returned = observable["Returned"] & (observable["DurationDays"] <= threshold + horizon)
        results.append({
            "ThresholdDays": threshold,
            "ReturnHorizonDays": horizon,
            "ReachedThreshold": len(reached_threshold),
            "ObservableSpells": len(observable),
            "InsufficientFollowUp": len(reached_threshold) - len(observable),
            "ReturnedWithinHorizon": returned.sum(),
            "ReturnProbability": returned.mean() * 100,
        })

conditional_return_probability = pd.DataFrame(results)
conditional_return_probability.round(2)

,ThresholdDays,ReturnHorizonDays,ReachedThreshold,ObservableSpells,InsufficientFollowUp,ReturnedWithinHorizon,ReturnProbability
0,30,30,18150,17402,748,5968,34.29
1,30,60,18150,16909,1241,8623,51.00
2,30,90,18150,16677,1473,10252,61.47
3,30,180,18150,16165,1985,12421,76.84
4,60,30,11433,10941,492,2655,24.27
5,60,60,11433,10709,724,4284,40.00
6,60,90,11433,10533,900,5292,50.24
7,60,180,11433,10041,1392,6801,67.73
8,90,30,8286,8054,232,1629,20.23
9,90,60,8286,7878,408,2637,33.47


## 19. Operational churn rule

This project uses a purchase-based rule designed for a non-contractual retail setting. A customer reaches churn after **60 consecutive days without a valid positive purchase**. Cancellation invoices remain useful customer interactions, but they do not reset purchase inactivity. A fully reversed purchase stops being valid once its matched full cancellation is known. Positive manual and zero-price invoices remain purchases under the approved cleaning rules unless they are fully reversed.

The prediction task is deliberately separate from the 60-day status rule. At each monthly `ReferenceDate`, the model scores every customer who is still active and predicts whether that customer will reach the 60-day inactivity boundary during the **next 30 days**. Customers already churned at the reference date are retained for descriptive population analysis but are not observations for this churn-prediction task. Reactivation is outside the current modeling scope.

This structure separates three concepts that must not be confused: the historical information available at the reference date, the 60-day business definition of churn, and the 30-day forecast horizon. It also permits a temporal validation gap based on the exact date at which each label becomes observable, with a maximum look-ahead of 30 days.

Methodological context: non-contractual customer inactivity is not directly observable as permanent departure, so the threshold is an operational decision rather than a universal market constant. The project therefore reports the 60-day rule explicitly and evaluates its consequences using the interval and return analyses above.

Methodological references: [Fader, Hardie and Shang, Customer-Base Analysis in a Discrete-Time Noncontractual Setting](https://www.brucehardie.com/papers/020/) and [Forecasting client retention, a machine-learning approach](https://www.sciencedirect.com/science/article/pii/S0969698919302668).